# Hyperparameter Tuning

**Topic:** Model Evaluation

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, HTML
from IPython.display import display, clear_output
from joblib import Parallel, delayed
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier)
from sklearn.model_selection import (train_test_split, cross_val_score,
    GridSearchCV, RandomizedSearchCV)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, roc_curve, auc, f1_score, precision_score, recall_score)
np.random.seed(42)
from tkh_utils import (PALETTE, FONT, base_layout, plot_confusion_matrix,
    plot_roc_curve, plot_feature_importance, plot_learning_curve)

housing = fetch_california_housing(as_frame=True)
X_reg, y_reg = housing.data, housing.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

try:
    from sklearn.datasets import fetch_openml
    _hd = fetch_openml(name='heart-disease', version=1, as_frame=True)
    X_cls = _hd.data
    y_cls = (_hd.target.astype(int) > 0).astype(int)
except Exception:
    from sklearn.datasets import load_breast_cancer as _lbc
    _bc = _lbc(as_frame=True)
    X_cls, y_cls = _bc.data, _bc.target
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42)

---
## What you'll explore

By the end of this session you will be able to:

- **Describe** the difference between grid search, random search, and Bayesian optimization
- **Interpret** a grid search heatmap to identify promising hyperparameter regions
- **Explain** why random search finds good hyperparameters with fewer evaluations than grid search

> **Tip:** Look at the grid search heatmap and find the column of `max_depth` values where performance starts to plateau. That plateau tells you when adding more depth stops helping — and where adding regularization becomes more important than adding capacity.

---
## How we got here

In `ml_concepts/16_what_are_hyperparameters.ipynb` you learned the difference between parameters (learned from data) and hyperparameters (set before training). This notebook is the practical companion: given a model with multiple hyperparameters, how do you search for the combination that produces the best generalization?

Hyperparameter tuning is always done with cross validation — never on a held-out test set — to avoid overfitting the hyperparameters themselves.

---
## Why this matters for data science

Default hyperparameters are almost never optimal for your specific dataset. A Random Forest with default `n_estimators=100` and `max_depth=None` might score R²=0.78; the same architecture with tuned hyperparameters might reach 0.84. That gap is often larger than the improvement from switching to a different algorithm entirely.

Tuning also reveals the *shape* of the performance landscape — flat regions where hyperparameters barely matter, sharp peaks where a specific combination is critical, and interactions between hyperparameters that would be invisible if you tuned them one at a time.

---
## Real-world example: Finding the best Random Forest for California Housing

We run a GridSearchCV over four values of `n_estimators` and four values of `max_depth`, using 3-fold CV. The resulting heatmap shows the performance landscape — and reveals whether adding more trees or more depth gives the bigger benefit.

In [2]:
param_grid = {
    'n_estimators': [10, 50, 100, 200],
    'max_depth': [3, 5, 10, 20],
}
gs = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid, cv=3, scoring='r2', n_jobs=-1,
)
gs.fit(X_reg_train, y_reg_train)

n_est_vals  = param_grid['n_estimators']
depth_vals  = [str(d) for d in param_grid['max_depth']]
results_mat = gs.cv_results_['mean_test_score'].reshape(
    len(param_grid['max_depth']), len(param_grid['n_estimators'])
)

best_row = list(param_grid['max_depth']).index(gs.best_params_['max_depth'])
best_col = param_grid['n_estimators'].index(gs.best_params_['n_estimators'])

layout = base_layout(
    title=f"Grid Search R² Heatmap — Best: n_est={gs.best_params_['n_estimators']}, "
          f"max_depth={gs.best_params_['max_depth']} (R²={gs.best_score_:.3f})",
    xaxis_title="n_estimators",
    yaxis_title="max_depth",
)
fig = go.Figure(layout=layout)
fig.add_trace(go.Heatmap(
    z=results_mat,
    x=[str(n) for n in n_est_vals],
    y=depth_vals,
    colorscale=[[0, PALETTE["background"]], [1, PALETTE["primary"]]],
    text=[[f"{v:.3f}" for v in row] for row in results_mat],
    texttemplate="%{text}",
    showscale=True,
))
fig.add_annotation(
    x=str(gs.best_params_['n_estimators']),
    y=str(gs.best_params_['max_depth']),
    text="Best",
    showarrow=True,
    arrowhead=2,
    font=dict(color=PALETTE["secondary"], size=13),
)
fig.show()

---
## Try it yourself

In [3]:
# Widget 1 — Grid Search Heatmap
# Uses a fixed 800-row subsample of the training data (not the full ~16.5k
# rows the real-world example above used) so that even the densest grid
# refits quickly enough to stay interactive — R² values here run lower than
# the real-world example's because of that smaller sample, not because the
# search itself is worse.
_rng1 = np.random.RandomState(42)
_idx1 = _rng1.choice(len(X_reg_train), size=800, replace=False)
_X_sub1 = X_reg_train.iloc[_idx1].values
_y_sub1 = y_reg_train.iloc[_idx1].values

def _fit_score1(n_est, depth):
    m = RandomForestRegressor(n_estimators=n_est, max_depth=depth, random_state=42, n_jobs=1)
    return cross_val_score(m, _X_sub1, _y_sub1, cv=3, scoring='r2', n_jobs=1).mean()

_grid_cache = {}

def _get_grid(density):
    if density not in _grid_cache:
        n_vals = sorted(set(int(v) for v in np.linspace(10, 200, density)))
        d_vals = sorted(set(int(v) for v in np.linspace(3, 20, density)))
        combos = [(n, d) for d in d_vals for n in n_vals]
        # Fitting each grid cell is independent of the others, unlike the
        # sequential Bayesian search in Widget 2 below — parallelizing across
        # cells (rather than inside each cell's own cross_val_score) is what
        # keeps even the densest grid under ~3 seconds.
        scores = Parallel(n_jobs=-1)(delayed(_fit_score1)(n, d) for n, d in combos)
        mat = np.array(scores).reshape(len(d_vals), len(n_vals))
        _grid_cache[density] = {"n_vals": n_vals, "d_vals": d_vals, "mat": mat}
    return _grid_cache[density]

out1 = Output()
explain_html1 = HTML()

density_slider = IntSlider(
    value=4, min=2, max=6, step=1,
    description="Grid density (values per hyperparameter):",
    style={"description_width": "260px"},
    layout=widgets.Layout(width="520px"),
    continuous_update=False,
)

def render_grid(change=None):
    density = density_slider.value
    data = _get_grid(density)
    n_vals, d_vals, mat = data["n_vals"], data["d_vals"], data["mat"]
    best_idx = np.unravel_index(np.argmax(mat), mat.shape)
    best_d, best_n = d_vals[best_idx[0]], n_vals[best_idx[1]]
    best_score = mat[best_idx]
    n_combos = len(n_vals) * len(d_vals)
    n_fits = n_combos * 3

    fig = go.Figure(layout=base_layout(
        title=f"Grid Search R² — {len(n_vals)}×{len(d_vals)} grid, cv=3 ({n_fits} model fits)",
        xaxis_title="n_estimators",
        yaxis_title="max_depth",
    ))
    fig.add_trace(go.Heatmap(
        z=mat, x=[str(n) for n in n_vals], y=[str(d) for d in d_vals],
        colorscale=[[0, PALETTE["background"]], [1, PALETTE["primary"]]],
        zmin=0.45, zmax=0.72,
        text=[[f"{v:.3f}" for v in row] for row in mat],
        texttemplate="%{text}", showscale=True,
    ))
    fig.add_annotation(
        x=str(best_n), y=str(best_d), text="Best", showarrow=True, arrowhead=2,
        font=dict(color=PALETTE["secondary"], size=13),
    )
    fig.update_layout(height=440)

    with out1:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    explain_html1.value = (
        f"<p style='max-width:720px'>At density={density}, grid search evaluates "
        f"<b>{len(n_vals)}×{len(d_vals)} = {n_combos}</b> combinations, each with 3-fold "
        f"cross-validation — <b>{n_fits} model fits total</b>. The best combination found "
        f"here is n_estimators={best_n}, max_depth={best_d} (R²={best_score:.3f}). Push the "
        f"density slider up and watch the fit count grow as density² × cv folds — that "
        f"squared growth is the exponential cost grid search is known for.</p>"
    )

density_slider.observe(render_grid, names="value")
display(VBox([density_slider, out1, explain_html1]))
render_grid()

In [4]:
# Widget 2 — Search Strategy Comparison
# An even smaller subsample and cv=2 than Widget 1 — Bayesian optimization is
# inherently sequential (each trial depends on the last, so its 50 evaluations
# cannot be parallelized the way Widget 1's grid cells were), so the per-fit
# cost has to come down further to keep the slowest strategy responsive.
_rng2 = np.random.RandomState(7)
_idx2 = _rng2.choice(len(X_reg_train), size=400, replace=False)
_X_sub2 = X_reg_train.iloc[_idx2].values
_y_sub2 = y_reg_train.iloc[_idx2].values

_N_EST_RANGE = (10, 100)
_DEPTH_RANGE = (3, 20)

def _fit_score2(n_est, depth):
    m = RandomForestRegressor(n_estimators=n_est, max_depth=depth, random_state=42, n_jobs=1)
    return cross_val_score(m, _X_sub2, _y_sub2, cv=2, scoring='r2', n_jobs=1).mean()

def _run_grid(n_evals):
    # Grid search cannot target an exact evaluation count — it fills a
    # rectangle evenly, so the actual count is side x side, close to n_evals
    # but rarely equal to it.
    side = max(2, round(np.sqrt(n_evals)))
    n_vals = sorted(set(int(v) for v in np.linspace(*_N_EST_RANGE, side)))
    d_vals = sorted(set(int(v) for v in np.linspace(*_DEPTH_RANGE, side)))
    combos = [(n, d) for n in n_vals for d in d_vals]
    scores = Parallel(n_jobs=-1)(delayed(_fit_score2)(n, d) for n, d in combos)
    return [(n, d, s) for (n, d), s in zip(combos, scores)]

def _run_random(n_evals):
    rng = np.random.RandomState(42)
    n_vals = rng.randint(_N_EST_RANGE[0], _N_EST_RANGE[1] + 1, size=n_evals)
    d_vals = rng.randint(_DEPTH_RANGE[0], _DEPTH_RANGE[1] + 1, size=n_evals)
    scores = Parallel(n_jobs=-1)(delayed(_fit_score2)(n, d) for n, d in zip(n_vals, d_vals))
    return [(int(n), int(d), s) for n, d, s in zip(n_vals, d_vals, scores)]

def _run_bayesian(n_evals):
    def _objective(trial):
        n_est = trial.suggest_int('n_estimators', *_N_EST_RANGE)
        depth = trial.suggest_int('max_depth', *_DEPTH_RANGE)
        return _fit_score2(n_est, depth)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(_objective, n_trials=n_evals)
    return [(t.params['n_estimators'], t.params['max_depth'], t.value) for t in study.trials]

_STRATEGY_FUNCS = {
    "Grid Search": _run_grid,
    "Random Search": _run_random,
    "Bayesian (TPE)": _run_bayesian,
}
_strategy_cache = {}

def _get_strategy_results(strategy, n_evals):
    key = (strategy, n_evals)
    if key not in _strategy_cache:
        _strategy_cache[key] = _STRATEGY_FUNCS[strategy](n_evals)
    return _strategy_cache[key]

out2 = Output()
explain_html2 = HTML()

strategy_dropdown = Dropdown(
    options=list(_STRATEGY_FUNCS.keys()), value="Grid Search",
    description="Strategy:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
)
n_evals_slider = IntSlider(
    value=16, min=5, max=50, step=1,
    description="Number of evaluations:",
    style={"description_width": "170px"},
    layout=widgets.Layout(width="460px"),
    continuous_update=False,
)

def render_strategy(change=None):
    strategy = strategy_dropdown.value
    n_evals = n_evals_slider.value
    results = _get_strategy_results(strategy, n_evals)
    ns = [r[0] for r in results]
    ds = [r[1] for r in results]
    scores = [r[2] for r in results]
    best_idx = int(np.argmax(scores))

    fig = go.Figure(layout=base_layout(
        title=f"{strategy} — {len(results)} evaluations (best R²={scores[best_idx]:.3f})",
        xaxis_title="n_estimators",
        yaxis_title="max_depth",
    ))
    fig.add_trace(go.Scatter(
        x=ns, y=ds, mode="markers",
        marker=dict(
            size=14, color=scores,
            colorscale=[[0, PALETTE["background"]], [1, PALETTE["primary"]]],
            cmin=0.2, cmax=0.7, showscale=True, colorbar=dict(title="R²"),
            line=dict(color="white", width=1),
        ),
        text=[f"n_est={n}, depth={d}<br>R²={s:.3f}" for n, d, s in zip(ns, ds, scores)],
        hoverinfo="text", name="Evaluated",
    ))
    fig.add_trace(go.Scatter(
        x=[ns[best_idx]], y=[ds[best_idx]], mode="markers",
        marker=dict(size=20, color=PALETTE["accent"], symbol="star",
                    line=dict(color="white", width=1)),
        name="Best found",
    ))
    fig.update_layout(
        xaxis=dict(range=[_N_EST_RANGE[0] - 5, _N_EST_RANGE[1] + 5]),
        yaxis=dict(range=[_DEPTH_RANGE[0] - 1, _DEPTH_RANGE[1] + 1]),
        height=460, showlegend=True,
    )

    with out2:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    if strategy == "Grid Search":
        note = (
            f"Grid search laid down a regular lattice of {len(results)} points — it cannot "
            f"target exactly {n_evals} evaluations, since it has to fill a rectangle evenly "
            f"rather than pick an arbitrary count."
        )
    elif strategy == "Random Search":
        note = (
            "Random search scattered points uniformly across the whole space — no lattice, "
            "so every evaluation lands somewhere a grid might have skipped entirely."
        )
    else:
        note = (
            "Bayesian optimization used every previous trial's result to decide where to "
            "look next, so later evaluations cluster around whichever region looked most "
            "promising instead of spreading out evenly."
        )

    explain_html2.value = (
        f"<p style='max-width:720px'>{note} Best found: n_estimators={ns[best_idx]}, "
        f"max_depth={ds[best_idx]} (R²={scores[best_idx]:.3f}). Switch strategies at the "
        f"same number of evaluations and compare how tightly the points cluster around the "
        f"best region versus how much of the space gets covered.</p>"
    )

strategy_dropdown.observe(render_strategy, names="value")
n_evals_slider.observe(render_strategy, names="value")
display(VBox([strategy_dropdown, n_evals_slider, out2, explain_html2]))
render_strategy()

---
## What's happening?

**Grid search** defines a discrete grid of hyperparameter values and evaluates every combination. It is exhaustive and guaranteed to find the best combination *in the grid*, but it scales exponentially — 4 values on each of 4 hyperparameters means 256 combinations, times k folds.

**Random search** samples randomly from the hyperparameter space. It covers a wider range with fewer evaluations, and for most problems it finds hyperparameters almost as good as grid search in 10–20% of the evaluations. The key insight: hyperparameter importance is unequal. If only 2 of 5 hyperparameters actually matter, grid search wastes most evaluations varying the irrelevant ones.

**Bayesian optimization** (Optuna, Hyperopt) treats hyperparameter tuning as a sequential decision problem. It builds a surrogate model of the performance landscape after each evaluation and uses that model to decide where to look next — spending more evaluations in promising regions.

---
## Reference

| Method | Strategy | Evaluations needed | Best for | Python library |
|---|---|---|---|---|
| Grid Search | All combinations | High | Small search spaces (≤ 3 hyperparams) | `sklearn.GridSearchCV` |
| Random Search | Random sampling | Medium | Wide search spaces, quick baseline | `sklearn.RandomizedSearchCV` |
| Bayesian (TPE) | Sequential model-based | Low | Large or expensive search spaces | `optuna` |
| Halving Grid | Progressive elimination | Medium-low | Large grids with limited compute | `sklearn.HalvingGridSearchCV` |

---
## Key takeaway

> **Grid search is thorough but expensive; random search finds near-optimal hyperparameters faster; Bayesian optimization is the professional's choice when each model fit is costly.**

---
*Next up: `06_model_selection.ipynb` — comparing different algorithm families and choosing the right one for your problem*